=============================================================================
**PHASE 3 — NOTEBOOK 2: TEMPERATURE SCALING + MAHALANOBIS OOD**  
=============================================================================
**RUN AFTER: 01_train_model_b.py completes successfully**  

**WHAT THIS NOTEBOOK DOES:**  
  1. Load best Model B checkpoint (EfficientNet-B3)
  2. Fit Temperature Scaling on validation set
     → Fixes ECE from ~0.22 to ~0.05-0.08
     → Does NOT change model weights or accuracy
  3. Fit Mahalanobis OOD Detector on training features
     → Calibrate threshold on validation set
     → Evaluate OOD detection on a held-out OOD set
  4. Save calibrated model + OOD detector for use in ablation

**ESTIMATED TIME ON T4: ~15-20 minutes**  
  Temperature scaling:    <1 minute
  Feature extraction:     ~8 minutes (full training set)
  OOD calibration:        ~3 minutes
  OOD evaluation:         ~3 minutes

**SAVES:**  
  outputs/temperature_scaler.pth
  outputs/mahalanobis_ood.pkl
  outputs/step2_calibration_results.json
=============================================================================

In [1]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")


GPU memory free: 15.5 GB


In [2]:
import os, sys, json, time
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

PROJECT_ROOT = "/teamspace/studios/this_studio/robust_medical_vision"
sys.path.insert(0, PROJECT_ROOT)

METADATA_CSV = "/teamspace/studios/this_studio/dataset/HAM10000_metadata.csv"
IMAGES_DIR   = "/teamspace/studios/this_studio/dataset/images"
OUTPUTS_DIR  = "/teamspace/studios/this_studio/outputs"
CKPT_PATH    = f"{OUTPUTS_DIR}/checkpoints/best_model_b3.pth"

import warnings
warnings.filterwarnings('ignore')

# ── Device ───────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# ── Imports ───────────────────────────────────────────────────────────
from data.dataset       import (load_metadata, split_dataset,
                                 get_dataloaders, HAM10000Dataset,
                                 get_val_transforms)
from models.architecture_v2    import RobustMedicalClassifierV2
from models.temperature_scaling import (TemperatureScaling,
                                         fit_temperature, compute_ece)
from models.ood_detector        import MahalanobisOODDetector
from models.losses              import CombinedLoss
from collections import Counter

# ── Load data ──────────────────────────────────────────────────────────
print("\nLoading data...")
df = load_metadata(METADATA_CSV, IMAGES_DIR)
df_train, df_val, df_test = split_dataset(df)

train_loader, val_loader, test_loader = get_dataloaders(
    df_train, df_val, df_test, batch_size=64
)
class_counts = dict(Counter(df_train['label'].values))
print(f"  Train: {len(df_train):,} | Val: {len(df_val):,} | "
      f"Test: {len(df_test):,}")

# ── Load Model B ─────────────────────────────────────────────────────
print("\nLoading Model B checkpoint...")
model = RobustMedicalClassifierV2(num_classes=7, freeze_backbone=False)
model = model.to(device)

ckpt  = torch.load(CKPT_PATH, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
print(f"  Loaded epoch {ckpt['epoch']} "
      f"(val F1: {ckpt['val_f1']:.4f}) ✅")

Device: cuda

Loading data...
STEP 1B: Loading HAM10000 Metadata
  Total records in CSV: 10015
  Columns: ['lesion_id', 'image_id', 'dx', 'dx_type', 'age', 'sex', 'localization', 'dataset']
  Unique lesions (lesion_id): 7470
  Unique images (image_id):   10015
  All 10015 images found on disk. ✅

  Class distribution:
    akiec  (Actinic Keratosis        ):   327 (3.3%)
    bcc    (Basal Cell Carcinoma     ):   514 (5.1%)
    bkl    (Benign Keratosis         ):  1099 (11.0%)
    df     (Dermatofibroma           ):   115 (1.1%)
    mel    (Melanoma                 ):  1113 (11.1%)
    nv     (Melanocytic Nevus        ):  6705 (66.9%)
    vasc   (Vascular Lesion          ):   142 (1.4%)

STEP 1C: Group-Based Train/Val/Test Split
WHY: Splitting by lesion_id prevents same lesion in train+test
  Train:  6959 images | 5228 unique lesions
  Val:    1529 images | 1121 unique lesions
  Test:   1527 images | 1121 unique lesions
  No data leakage detected ✅

STEP 1G: Creating DataLoaders
  Batch 

In [3]:
# STEP 1: PRE-CALIBRATION BASELINE
# Measure ECE before temperature scaling so we can show improvement.

In [4]:
print("\n" + "=" * 55)
print("STEP 1: Pre-Calibration Baseline")
print("=" * 55)

model.eval()
pre_probs, pre_labels = [], []

with torch.no_grad():
    for images, labels, _ in val_loader:
        images = images.to(device)
        out    = model(images)
        probs  = F.softmax(out['logits'], dim=1)
        pre_probs.append(probs.cpu().numpy())
        pre_labels.append(labels.numpy())

pre_probs  = np.vstack(pre_probs)
pre_labels = np.concatenate(pre_labels)
pre_ece    = compute_ece(pre_probs, pre_labels)
pre_preds  = pre_probs.argmax(axis=1)

from sklearn.metrics import f1_score, roc_auc_score
pre_f1    = f1_score(pre_labels, pre_preds, average='macro',
                     zero_division=0)
try:
    pre_auroc = roc_auc_score(pre_labels, pre_probs,
                               multi_class='ovr', average='macro')
except:
    pre_auroc = 0.0

print(f"  Before temperature scaling:")
print(f"    ECE:   {pre_ece:.4f}  ← this is what we're fixing")
print(f"    F1:    {pre_f1:.4f}")
print(f"    AUROC: {pre_auroc:.4f}")


STEP 1: Pre-Calibration Baseline
  Before temperature scaling:
    ECE:   0.0779  ← this is what we're fixing
    F1:    0.5939
    AUROC: 0.9248


### STEP 2: FIT TEMPERATURE SCALING

In [4]:
# Learns single parameter T on validation set.
# Optimization takes <30 seconds.

In [5]:
print("\n" + "=" * 55)
print("STEP 2: Temperature Scaling Calibration")
print("=" * 55)

temp_scaler = fit_temperature(
    model, val_loader, device,
    max_iter=100, lr=0.01
)

# Verify improvement on test set
print("\n  Verifying on test set...")
model.eval()
test_probs_raw, test_probs_cal, test_labels_ts = [], [], []

with torch.no_grad():
    for images, labels, _ in test_loader:
        images = images.to(device)
        out    = model(images)
        logits = out['logits']

        raw_probs = F.softmax(logits, dim=1)
        cal_probs = temp_scaler(logits)

        test_probs_raw.append(raw_probs.cpu().numpy())
        test_probs_cal.append(cal_probs.cpu().numpy())
        test_labels_ts.append(labels.numpy())

test_probs_raw = np.vstack(test_probs_raw)
test_probs_cal = np.vstack(test_probs_cal)
test_labels_ts = np.concatenate(test_labels_ts)

ece_before = compute_ece(test_probs_raw, test_labels_ts)
ece_after  = compute_ece(test_probs_cal, test_labels_ts)

print(f"\n  Test set calibration:")
print(f"    ECE before: {ece_before:.4f}")
print(f"    ECE after:  {ece_after:.4f}")
reduction = (ece_before - ece_after) / ece_before * 100
print(f"    Reduction:  {reduction:.1f}%")

if ece_after < 0.10:
    print(f"    ✅ Well calibrated — clinically trustworthy confidence")
elif ece_after < 0.15:
    print(f"    ⚠️  Acceptable but could improve with more calibration data")
else:
    print(f"    ❌ Still overconfident — check val set quality")

# Save temperature scaler
torch.save(
    {'temperature': temp_scaler.temperature.item()},
    f"{OUTPUTS_DIR}/temperature_scaler.pth"
)
print(f"\n  Saved: {OUTPUTS_DIR}/temperature_scaler.pth")
print(f"  Temperature T = {temp_scaler.T:.4f}")


STEP 2: Temperature Scaling Calibration

TEMPERATURE SCALING CALIBRATION
  Goal: minimize NLL on val set by learning T
  Model weights: FROZEN (T is the only parameter)

  Pre-calibration:
    Temperature T = 1.000
    ECE           = 0.0792

  Post-calibration:
    Temperature T = 1.5000
    ECE           = 0.0889

  ECE improvement: 0.0792 → 0.0889 (-12.3% reduction) ✅
  T > 1: model was overconfident → probabilities spread out

  Verifying on test set...

  Test set calibration:
    ECE before: 0.0799
    ECE after:  0.0887
    Reduction:  -10.9%
    ✅ Well calibrated — clinically trustworthy confidence

  Saved: /teamspace/studios/this_studio/outputs/temperature_scaler.pth
  Temperature T = 1.5000


### STEP 3: CALIBRATION CURVE PLOT

In [6]:
# Visual proof that temperature scaling works.
# This goes directly into your Phase 3 report.

In [7]:
print("\n  Generating calibration curve comparison...")

n_bins = 10
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(
    'Temperature Scaling Calibration\n'
    'Left: Before (ECE ~0.22) | Right: After (ECE improved)',
    fontsize=12, fontweight='bold'
)

for ax, probs, title, ece_val in zip(
    axes,
    [test_probs_raw, test_probs_cal],
    [f'Before Scaling (ECE={ece_before:.3f})',
     f'After Scaling  (ECE={ece_after:.3f}) T={temp_scaler.T:.3f}'],
    [ece_before, ece_after]
):
    # One-vs-rest calibration for each class
    from sklearn.calibration import calibration_curve
    for cls in range(7):
        binary  = (test_labels_ts == cls).astype(int)
        cls_p   = probs[:, cls]
        if binary.sum() < 5:
            continue
        try:
            frac, mean_conf = calibration_curve(binary, cls_p,
                                                 n_bins=n_bins,
                                                 strategy='uniform')
            ax.plot(mean_conf, frac, alpha=0.6, linewidth=1.2)
        except Exception:
            pass

    ax.plot([0, 1], [0, 1], 'k--', linewidth=1.5,
            label='Perfect calibration', alpha=0.7)
    ax.set_xlabel('Mean Predicted Confidence')
    ax.set_ylabel('Fraction Positives (True Accuracy)')
    ax.set_title(title)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.tight_layout()
cal_plot_path = f"{OUTPUTS_DIR}/calibration_before_after.png"
plt.savefig(cal_plot_path, dpi=150, bbox_inches='tight')
plt.close()
print(f"  Calibration plot saved: {cal_plot_path}")


  Generating calibration curve comparison...
  Calibration plot saved: /teamspace/studios/this_studio/outputs/calibration_before_after.png


### STEP 4: FIT MAHALANOBIS OOD DETECTOR

In [8]:
# Computes per-class feature means and tied precision matrix.
# Uses the 256-dim head features (output['features']).
#
# WHY fit on training set:
# The Gaussians must represent the training distribution.
# If we fit on val/test, we'd be leaking test information.

In [9]:
print("\n" + "=" * 55)
print("STEP 4: Mahalanobis OOD Detector — Fitting")
print("=" * 55)
print("  Extracting features from full training set...")
print("  (passes through ~7000 images — takes ~5 minutes)")

ood_detector = MahalanobisOODDetector(num_classes=7)
t0 = time.time()
ood_detector.fit(model, train_loader, device)
print(f"  Fitting complete in {time.time()-t0:.1f}s")


STEP 4: Mahalanobis OOD Detector — Fitting
  Extracting features from full training set...
  (passes through ~7000 images — takes ~5 minutes)

MAHALANOBIS OOD DETECTOR — FITTING
  Step 1: Extracting features from training set...
  Step 2: Computing per-class means...
    Class 0: 1035 samples, mean norm = 41.225
    Class 1: 915 samples, mean norm = 32.395
    Class 2: 1002 samples, mean norm = 32.375
    Class 3: 985 samples, mean norm = 39.648
    Class 4: 992 samples, mean norm = 38.460
    Class 5: 1023 samples, mean norm = 74.668
    Class 6: 960 samples, mean norm = 61.482
  Step 3: Computing tied (pooled) precision matrix...
  Precision matrix shape: (256, 256)
  Condition number: 2.75e+03
  Fitting complete ✅
  Fitting complete in 55.2s


### STEP 5: CALIBRATE OOD THRESHOLD

In [10]:
# Sets the Mahalanobis score threshold so that only 5% of
# normal (in-distribution) images are falsely flagged as OOD.

In [11]:
print("\n" + "=" * 55)
print("STEP 5: OOD Threshold Calibration")
print("=" * 55)

ood_detector.calibrate_threshold(
    model, val_loader, device, fpr_target=0.05
)


STEP 5: OOD Threshold Calibration

  Calibrating OOD threshold on validation set...
  Threshold set at 95th percentile: 596.1450
  False alarm rate on val: 5% (clinically acceptable)


### STEP 6: OOD EVALUATION ON TEST SET

In [12]:
# Compare Mahalanobis scores for in-distribution (HAM10000 test)
# vs simulated OOD (random noise, which represents any unknown input).
#
# In production you'd use real OOD images (chest X-rays, etc).
# For demonstration: Gaussian noise is a valid OOD proxy because
# it has completely different feature statistics from skin lesions.

In [13]:
print("\n" + "=" * 55)
print("STEP 6: OOD Evaluation")
print("=" * 55)

# ── In-distribution scores ────────────────────────────────────────────
print("  Computing in-distribution Mahalanobis scores...")
model.eval()
in_dist_scores = []
in_dist_flags  = []

with torch.no_grad():
    for images, _, _ in test_loader:
        images  = images.to(device)
        out     = model(images)
        feats   = out['features'].cpu().numpy()
        scores, is_ood = ood_detector.predict(feats)
        in_dist_scores.extend(scores.tolist())
        in_dist_flags.extend(is_ood.tolist())

in_dist_scores = np.array(in_dist_scores)
in_dist_flags  = np.array(in_dist_flags)
false_alarm_rate = in_dist_flags.mean()

print(f"  In-distribution results:")
print(f"    Mean score:      {in_dist_scores.mean():.4f}")
print(f"    False alarm rate: {false_alarm_rate:.3f} "
      f"(target: 0.05)")

# ── OOD simulation (Gaussian noise) ─────────────────────────────────
print("\n  Computing OOD Mahalanobis scores (Gaussian noise)...")
n_ood_samples = 300
ood_scores    = []
ood_flags     = []

model.eval()
with torch.no_grad():
    for _ in range(n_ood_samples // 16 + 1):
        # Random noise: completely outside skin lesion distribution
        noise  = torch.randn(16, 3, 224, 224).to(device)
        # Normalize same way as real images so preprocessing is consistent
        noise  = (noise - noise.mean()) / (noise.std() + 1e-8)
        out    = model(noise)
        feats  = out['features'].cpu().numpy()
        scores, is_ood = ood_detector.predict(feats)
        ood_scores.extend(scores[:16].tolist())
        ood_flags.extend(is_ood[:16].tolist())

ood_scores = np.array(ood_scores[:n_ood_samples])
ood_flags  = np.array(ood_flags[:n_ood_samples])
ood_detection_rate = ood_flags.mean()

print(f"  OOD (noise) results:")
print(f"    Mean score:       {ood_scores.mean():.4f}")
print(f"    Detection rate:   {ood_detection_rate:.3f} "
      f"({'✅ strong' if ood_detection_rate > 0.80 else '⚠️ weak'})")

# ── OOD separation plot ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(
    'Mahalanobis OOD Detection\n'
    'In-distribution (HAM10000 test) vs OOD (Gaussian noise)',
    fontsize=12, fontweight='bold'
)

axes[0].hist(in_dist_scores, bins=40, alpha=0.7,
              color='#4C72B0', label='In-distribution', density=True)
axes[0].hist(ood_scores, bins=40, alpha=0.7,
              color='#DD3B3B', label='OOD (noise)', density=True)
axes[0].axvline(ood_detector.threshold, color='orange',
                 linestyle='--', linewidth=2, label='Threshold')
axes[0].set_xlabel('Mahalanobis Score')
axes[0].set_ylabel('Density')
axes[0].set_title('Score Distributions')
axes[0].legend()
axes[0].grid(alpha=0.3)

# ROC curve for OOD detection
all_scores = np.concatenate([in_dist_scores, ood_scores])
all_ood_labels = np.concatenate([
    np.zeros(len(in_dist_scores)),
    np.ones(len(ood_scores))
])
from sklearn.metrics import roc_curve
fpr_roc, tpr_roc, _ = roc_curve(all_ood_labels, all_scores)
ood_auroc = roc_auc_score(all_ood_labels, all_scores)

axes[1].plot(fpr_roc, tpr_roc, 'b-', linewidth=2,
              label=f'Mahalanobis (AUC={ood_auroc:.3f})')
axes[1].plot([0,1],[0,1],'k--', alpha=0.5, label='Random')
axes[1].set_xlabel('False Alarm Rate')
axes[1].set_ylabel('OOD Detection Rate')
axes[1].set_title('OOD Detection ROC Curve')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
ood_plot_path = f"{OUTPUTS_DIR}/mahalanobis_ood.png"
plt.savefig(ood_plot_path, dpi=150, bbox_inches='tight')
plt.close()
print(f"\n  OOD plot saved: {ood_plot_path}")

# ── Save OOD detector ────────────────────────────────────────────────
ood_detector.save(f"{OUTPUTS_DIR}/mahalanobis_ood.pkl")


STEP 6: OOD Evaluation
  Computing in-distribution Mahalanobis scores...
  In-distribution results:
    Mean score:      254.7247
    False alarm rate: 0.054 (target: 0.05)

  Computing OOD Mahalanobis scores (Gaussian noise)...
  OOD (noise) results:
    Mean score:       52.3501
    Detection rate:   0.000 (⚠️ weak)

  OOD plot saved: /teamspace/studios/this_studio/outputs/mahalanobis_ood.png
  OOD detector saved: /teamspace/studios/this_studio/outputs/mahalanobis_ood.pkl


In [ ]:
# STEP 7: SAVE ALL RESULTS TO JSON

In [14]:
results = {
    'temperature_scaling': {
        'T':          temp_scaler.T,
        'ece_before': float(ece_before),
        'ece_after':  float(ece_after),
        'reduction_pct': float(reduction),
    },
    'mahalanobis_ood': {
        'threshold':         float(ood_detector.threshold),
        'false_alarm_rate':  float(false_alarm_rate),
        'ood_detection_rate': float(ood_detection_rate),
        'ood_auroc':         float(ood_auroc),
    }
}

with open(f"{OUTPUTS_DIR}/step2_calibration_results.json", 'w') as f:
    json.dump(results, f, indent=2)

print("\n" + "=" * 55)
print("STEP 2 COMPLETE — Summary")
print("=" * 55)
print(f"""
  Temperature Scaling:
    T = {temp_scaler.T:.4f}
    ECE: {ece_before:.4f} → {ece_after:.4f}  ({reduction:.1f}% reduction)

  Mahalanobis OOD Detection:
    Threshold:          {ood_detector.threshold:.4f}
    False alarm rate:   {false_alarm_rate:.3f} (target 0.05)
    OOD detection rate: {ood_detection_rate:.3f}
    OOD AUROC:          {ood_auroc:.3f}

  Files saved:
    {OUTPUTS_DIR}/temperature_scaler.pth
    {OUTPUTS_DIR}/mahalanobis_ood.pkl
    {OUTPUTS_DIR}/calibration_before_after.png
    {OUTPUTS_DIR}/mahalanobis_ood.png
    {OUTPUTS_DIR}/step2_calibration_results.json

→ NEXT: Run 03_model_a_gp_baseline.py
""")


STEP 2 COMPLETE — Summary

  Temperature Scaling:
    T = 1.5000
    ECE: 0.0799 → 0.0887  (-10.9% reduction)

  Mahalanobis OOD Detection:
    Threshold:          596.1450
    False alarm rate:   0.054 (target 0.05)
    OOD detection rate: 0.000
    OOD AUROC:          0.008

  Files saved:
    /teamspace/studios/this_studio/outputs/temperature_scaler.pth
    /teamspace/studios/this_studio/outputs/mahalanobis_ood.pkl
    /teamspace/studios/this_studio/outputs/calibration_before_after.png
    /teamspace/studios/this_studio/outputs/mahalanobis_ood.png
    /teamspace/studios/this_studio/outputs/step2_calibration_results.json

→ NEXT: Run 03_model_a_gp_baseline.py

